In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import json

import math
from torch.amp import autocast
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model, TaskType
import esm
from typing import List, Dict, Tuple, Optional, Any

## Strcuture embeddings in Stage 1

In [2]:
class ProjectionHead(nn.Module):
    """Projection head to map embeddings to contrastive learning space"""
    
    def __init__(self, input_dim, hidden_dim=1024, output_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

In [3]:
class ContrastiveModel(nn.Module):
    """Contrastive learning model for drug-target alignment"""
    
    def __init__(self, drug_dim, target_dim, hidden_dim=1024, output_dim=512):
        super().__init__()
        self.drug_projector = ProjectionHead(drug_dim, hidden_dim, output_dim)
        self.target_projector = ProjectionHead(target_dim, hidden_dim, output_dim)
        
    def forward(self, drug_embeds, target_embeds):
        drug_proj = self.drug_projector(drug_embeds)
        target_proj = self.target_projector(target_embeds)
        return drug_proj, target_proj

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ContrastiveModel(drug_dim = 384, target_dim = 1280, hidden_dim=1024, output_dim=512)
model.load_state_dict(torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/result/1229_0/best_model.pt", map_location=device))
model.to(device)
model.eval()

/scratch/slurm-1562169/ipykernel_3726456/480059224.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/C

ContrastiveModel(
  (drug_projector): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=384, out_features=1024, bias=True)
      (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=1024, out_features=512, bias=True)
    )
  )
  (target_projector): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=1280, out_features=1024, bias=True)
      (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=1024, out_features=512, bias=True)
    )
  )
)

In [6]:
smile_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/SMILE_embedding.pt", map_location=device)
protein_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/protein_embedding.pt", map_location=device)

/scratch/slurm-1562169/ipykernel_3726456/3758105350.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  smile_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrasti

In [7]:
new_smile_embeddings = {}
for smile, emb in smile_dict.items():
    x = emb.unsqueeze(0).to(device)
    with torch.no_grad():
        projected = model.drug_projector(x)
    new_smile_embeddings[smile] = projected.squeeze(0).cpu()

new_protein_embeddings = {}
for seq, emb in protein_dict.items():
    x = emb.unsqueeze(0).to(device)
    with torch.no_grad():
        projected = model.target_projector(x)
    new_protein_embeddings[seq] = projected.squeeze(0).cpu()

In [8]:
print("SMILE embedding shape:", list(new_smile_embeddings.values())[0].shape)
print("Protein embedding shape:", list(new_protein_embeddings.values())[0].shape)

SMILE embedding shape: torch.Size([512])
Protein embedding shape: torch.Size([512])


In [19]:
torch.save(new_smile_embeddings, "/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_smile_embeddings.pt")
torch.save(new_protein_embeddings, "/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_protein_embeddings.pt")

## Image embeddings in Stage 1

In [2]:
class ProjectionHead(nn.Module):
    """Deeper MLP projection head for contrastive learning."""
    
    def __init__(self, input_dim, hidden_dim=1204, output_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim // 2, output_dim)
        )
    
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


In [3]:
class ContrastiveModel(nn.Module):
    """Contrastive learning model for drug-target alignment"""
    
    def __init__(self, drug_dim, target_dim, hidden_dim=1024, output_dim=512):
        super().__init__()
        self.drug_projector = ProjectionHead(drug_dim, hidden_dim, output_dim)
        self.target_projector = ProjectionHead(target_dim, hidden_dim, output_dim)
        
    def forward(self, drug_embeds, target_embeds):
        drug_proj = self.drug_projector(drug_embeds)
        target_proj = self.target_projector(target_embeds)
        return drug_proj, target_proj

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ContrastiveModel(drug_dim = 737, target_dim = 599, hidden_dim=1024, output_dim=512)
model.load_state_dict(torch.load("/ix1/ychiu/yil346/DTI_v2/Image/contrastive_learning/result/1231/best_model.pt", map_location=device))
model.to(device)
model.eval()

/scratch/slurm-1562169/ipykernel_3728455/3389539542.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/ix1/ychiu/yil346/DTI_v2/Image/cont

ContrastiveModel(
  (drug_projector): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=737, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (6): ReLU()
      (7): Dropout(p=0.1, inplace=False)
      (8): Linear(in_features=512, out_features=512, bias=True)
    )
  )
  (target_projector): ProjectionHead(
    (net): Sequential(
      (0): Linear(in_features=599, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (6): ReLU()
      (7): Dropout(p=0.1, inplace=False)
      (8): Linear(in_features=

In [5]:
img_drug_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Image/contrastive_learning/data/embeddings/All_compound_image.pt", map_location=device)
img_protein_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Image/contrastive_learning/data/embeddings/All_CRISPR_image.pt", map_location=device)

/scratch/slurm-1562169/ipykernel_3728455/2598856589.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img_drug_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Image/contrastiv

In [6]:
new_img_drug_embeddings = {}
for smile, emb in img_drug_dict.items():
    x = emb.unsqueeze(0).to(device)
    with torch.no_grad():
        projected = model.drug_projector(x)
    new_img_drug_embeddings[smile] = projected.squeeze(0).cpu()

new_img_protein_embeddings = {}
for seq, emb in img_protein_dict.items():
    x = emb.unsqueeze(0).to(device)
    with torch.no_grad():
        projected = model.target_projector(x)
    new_img_protein_embeddings[seq] = projected.squeeze(0).cpu()

In [7]:
print("SMILE embedding shape:", list(new_img_drug_embeddings.values())[0].shape)
print("Protein embedding shape:", list(new_img_protein_embeddings.values())[0].shape)

SMILE embedding shape: torch.Size([512])
Protein embedding shape: torch.Size([512])


In [10]:
print(img_drug_dict['BrC1=CCCN(Cc2ccccn2)C1'][:5])
print(new_img_drug_embeddings['BrC1=CCCN(Cc2ccccn2)C1'][:5])

tensor([-0.0109,  0.9501, -0.5354,  0.1131, -1.0030], device='cuda:0')
tensor([-0.0955,  0.0261,  0.0010,  0.0633,  0.0269])


In [11]:
torch.save(new_img_drug_embeddings, "/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_img_drug_embeddings.pt")
torch.save(new_img_protein_embeddings, "/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_img_protein_embeddings.pt")

## LoRA Structure Stage 1 Embeddings (1230_1: ChemBERTa is all trained not LoRA)

In [2]:
chem_tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-77M-MTR")
chem_model = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR",trust_remote_code=True,use_safetensors=True)

#esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm_model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
batch_converter = alphabet.get_batch_converter()

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
def load_dti_model_for_inference(ckpt_path, device="cuda"):

    chem_model = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR", trust_remote_code=True, use_safetensors=True)

    esm_model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    esm_repr_layer = esm_model.num_layers
    esm_hidden = esm_model.embed_dim
    chem_hidden = chem_model.config.hidden_size

    esm_model = inject_lora_into_esm(
        esm_model,
        target_substrings=["q_proj", "k_proj", "v_proj", "out_proj"],
        r=4,
        alpha=16,
        dropout=0.05,
        layers_last_k=5,
    )
    esm_model = make_esm_lora_only(esm_model)

    dti_model = DTIContrastiveModel(
        chem_model=chem_model,
        esm_model=esm_model,
        alphabet=alphabet,
        chem_hidden=chem_hidden,
        esm_hidden=esm_hidden,
        esm_repr_layer=esm_repr_layer,
        proj_hidden=1024,
        proj_out=512,
        debug_assert_valid_tokens=False,
        use_cls_pool_for_protein=True,
    ).to(device)

    state = torch.load(ckpt_path, map_location="cpu")
    dti_model.load_state_dict(state, strict=True)

    for p in dti_model.parameters():
        p.requires_grad = False

    dti_model.eval()
    return dti_model, alphabet, batch_converter

In [4]:
class LoRALinear(nn.Module):
    """
    Wrap a frozen nn.Linear with a trainable low-rank update:
      y = x W^T + b + (alpha/r) * x (BA)^T
    """
    def __init__(self, base: nn.Linear, r: int = 4, alpha: int = 8, dropout: float = 0.05):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base = base
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.in_features = base.in_features
        self.out_features = base.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        self.dropout = nn.Dropout(dropout) if dropout and dropout > 0 else nn.Identity()

        self.A = nn.Linear(self.in_features, r, bias=False)
        self.B = nn.Linear(r, self.out_features, bias=False)

        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = self.B(self.A(self.dropout(x))) * self.scaling
        return base_out + lora_out

def _get_parent_module(root: nn.Module, module_name: str) -> Tuple[nn.Module, str]:
    parts = module_name.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]

def inject_lora_into_esm(
    esm_model: nn.Module,
    target_substrings: List[str],
    r: int = 4,
    alpha: int = 8,
    dropout: float = 0.05,
    layers_last_k: Optional[int] = None,
) -> nn.Module:

    # Freeze everything first
    for p in esm_model.parameters():
        p.requires_grad = False

    # Figure out allowed layer indices if requested
    allowed_layer_prefixes = None
    if layers_last_k is not None:
        # ESM2 (FAIR) commonly has .layers as ModuleList of length L
        if hasattr(esm_model, "layers") and isinstance(esm_model.layers, nn.ModuleList):
            L = len(esm_model.layers)
            start = max(0, L - layers_last_k)
            allowed_layer_prefixes = [f"layers.{i}." for i in range(start, L)]
        else:
            print("[WARN] layers_last_k provided but esm_model has no .layers ModuleList; ignoring.")

    replaced = 0
    # named_modules gives also wrappers; we want only leaf Linear modules
    for name, module in list(esm_model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue

        if allowed_layer_prefixes is not None:
            if not any(name.startswith(pref) for pref in allowed_layer_prefixes):
                continue

        if any(s in name for s in target_substrings):
            parent, attr = _get_parent_module(esm_model, name)
            wrapped = LoRALinear(module, r=r, alpha=alpha, dropout=dropout)
            setattr(parent, attr, wrapped)
            replaced += 1

    if replaced == 0:
        print("[WARN] No Linear layers matched for LoRA injection.")
        print("[INFO] Available module names:")
        for name, module in esm_model.named_modules():
            if isinstance(module, nn.Linear):
                print(f"  - {name}")
    else:
        print(f"[INFO] Injected LoRA into {replaced} Linear layers of ESM.")
        
    return esm_model

def make_esm_lora_only(esm_model):
    # Freeze everything
    for p in esm_model.parameters():
        p.requires_grad = False

    # Unfreeze only LoRA params (your LoRALinear uses ".A." and ".B.")
    for n, p in esm_model.named_parameters():
        if ".A." in n or ".B." in n:
            p.requires_grad = True

    # Safety check
    bad = [n for n, p in esm_model.named_parameters()
           if p.requires_grad and not (".A." in n or ".B." in n)]
    if bad:
        print("Still-trainable (should be frozen):", bad[:20])
    else:
        print("✅ ESM2 is LoRA-only now.")

    return esm_model

In [5]:
class ProjectorMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 1024, out_dim: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [6]:
class DTIContrastiveModel(nn.Module):
    """
    Two-tower encoders (ChemBERTa + ESM2) -> pooled embeddings -> projectors -> normalized z's.

    Key fixes vs your earlier version:
      - alphabet and esm_repr_layer are passed to __init__ (no globals / no NameError risk)
      - forward() does NOT take alphabet/esm_repr_layer/esm_padding_mask (no redundancy / fewer bugs)
      - padding mask is computed inside forward() from esm_tokens using alphabet.padding_idx
      - pooling excludes PAD + CLS + EOS safely (EOS anywhere, even if truncated)
      - optional debug assert to catch empty valid masks
    """
    def __init__(
        self,
        chem_model: nn.Module,
        esm_model: nn.Module,
        alphabet,
        chem_hidden: int,
        esm_hidden: int,
        esm_repr_layer: int = 30,
        proj_hidden: int = 1024,
        proj_out: int = 512,
        debug_assert_valid_tokens: bool = False,
        use_cls_pool_for_protein: bool = True,
    ):
        super().__init__()
        self.chem_model = chem_model
        self.esm_model = esm_model
        self.alphabet = alphabet
        self.esm_repr_layer = esm_repr_layer

        self.debug_assert_valid_tokens = debug_assert_valid_tokens
        self.use_cls_pool_for_protein = use_cls_pool_for_protein

        self.drug_proj = ProjectorMLP(chem_hidden, proj_hidden, proj_out)
        self.prot_proj = ProjectorMLP(esm_hidden, proj_hidden, proj_out)

    def forward(
        self,
        chem_inputs: Dict[str, torch.Tensor],
        esm_tokens: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        
        # -----------------
        # ChemBERTa forward
        # -----------------
        chem_out = self.chem_model(**chem_inputs)
        # RoBERTa-like models expose last_hidden_state: [B, T, H]
        drug_emb = chem_out.last_hidden_state[:, 0, :]  # CLS pooling

        # ------------
        # ESM2 forward
        # ------------
        esm_out = self.esm_model(
            esm_tokens,
            repr_layers=[self.esm_repr_layer],
            return_contacts=False
        )
        reps = esm_out["representations"][self.esm_repr_layer]  # [B, T, H]

        # Pool protein representation
        if self.use_cls_pool_for_protein:
            # Simple + padding-safe (but sometimes slightly worse than mean pooling)
            prot_emb = reps[:, 0, :]
        else:
            # Mean pool over real residues: exclude PAD + CLS + EOS
            pad_mask = esm_tokens.eq(self.alphabet.padding_idx)  # True for PAD
            mask = ~pad_mask                                      # True for non-PAD
            mask[:, 0] = False                                    # drop CLS token
            mask &= ~esm_tokens.eq(self.alphabet.eos_idx)         # drop EOS anywhere

            if self.debug_assert_valid_tokens and self.training:
                # Ensure each sequence has at least one valid residue token
                if not torch.all(mask.sum(dim=1) > 0):
                    bad = (mask.sum(dim=1) == 0).nonzero(as_tuple=True)[0].tolist()
                    raise ValueError(f"Found {len(bad)} sequences with 0 valid tokens after masking. Indices: {bad[:10]}")

            denom = mask.sum(dim=1).clamp(min=1).unsqueeze(-1)    # [B,1]
            prot_emb = (reps * mask.unsqueeze(-1)).sum(dim=1) / denom  # [B,H]

        # --------------------
        # Project & normalize
        # --------------------
        drug_z = F.normalize(self.drug_proj(drug_emb), dim=1)
        prot_z = F.normalize(self.prot_proj(prot_emb), dim=1)

        return drug_z, prot_z

In [7]:
ckpt_path = "/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/result/1230_2/best_model.pt"
device = "cuda"
dti_model, alphabet, batch_converter = load_dti_model_for_inference(ckpt_path, device=device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] Injected LoRA into 20 Linear layers of ESM.
✅ ESM2 is LoRA-only now.


/scratch/slurm-1571900/ipykernel_3502346/2407048730.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


In [8]:
smile_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/SMILE_embedding.pt", map_location=device)
protein_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/protein_embedding.pt", map_location=device)

unique_smiles = list(smile_dict.keys())
unique_proteins = list(protein_dict.keys())

/scratch/slurm-1571900/ipykernel_3502346/2518953623.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  protein_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contras

In [9]:
@torch.no_grad()
def embed_all_smiles(
    dti_model,
    smiles_list,
    chem_tokenizer,
    device="cuda",
    batch_size=256,
    max_drug_tokens=1024,
    use_amp=True,
):
    dti_model.eval()
    use_cuda = device.startswith("cuda")
    out = []

    for i in tqdm(range(0, len(smiles_list), batch_size), desc="Embed SMILES"):
        batch = smiles_list[i:i+batch_size]

        chem_inputs = chem_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_drug_tokens,
        )
        chem_inputs = {k: v.to(device) for k, v in chem_inputs.items()}

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(use_amp and use_cuda)):
            chem_out = dti_model.chem_model(**chem_inputs)
            drug_emb = chem_out.last_hidden_state[:, 0, :]  # CLS
            drug_z = F.normalize(dti_model.drug_proj(drug_emb), dim=1)

        out.append(drug_z.cpu())

    return torch.cat(out, dim=0)

In [10]:
drug_z_u = embed_all_smiles(
    dti_model, unique_smiles, chem_tokenizer,
    device=device,
    batch_size=128,
    max_drug_tokens=1024,
)
print(drug_z_u.shape)


Embed SMILES: 100%|██████████| 565/565 [00:10<00:00, 55.64it/s]

torch.Size([72227, 512])


In [9]:
@torch.no_grad()
def embed_all_proteins(
    dti_model,
    protein_list,
    batch_converter,
    device="cuda",
    batch_size=32,
    max_protein_tokens=1024,
    use_amp=True,
):
    dti_model.eval()
    use_cuda = device.startswith("cuda")
    out = []

    for i in tqdm(range(0, len(protein_list), batch_size), desc="Embed proteins"):
        batch = protein_list[i:i+batch_size]

        esm_data = [(str(j), seq) for j, seq in enumerate(batch)]
        _, _, tokens = batch_converter(esm_data)

        # truncate defensively
        if tokens.size(1) > max_protein_tokens:
            tokens = tokens[:, :max_protein_tokens]

        tokens = tokens.to(device)

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(use_amp and use_cuda)):
            esm_out = dti_model.esm_model(tokens, repr_layers=[dti_model.esm_repr_layer], return_contacts=False)
            reps = esm_out["representations"][dti_model.esm_repr_layer]

            # CLS pooling (matches your training)
            prot_emb = reps[:, 0, :]
            prot_z = F.normalize(dti_model.prot_proj(prot_emb), dim=1)

        out.append(prot_z.cpu())

    return torch.cat(out, dim=0)


In [10]:
prot_z_u = embed_all_proteins(
    dti_model, unique_proteins, batch_converter,
    device=device,
    batch_size=8,  # keep small for ESM
    max_protein_tokens=1024,
)
print(prot_z_u.shape)

Embed proteins: 100%|██████████| 1007/1007 [03:40<00:00,  4.57it/s]

torch.Size([8054, 512])


In [11]:
# Ensure CPU tensors (recommended for saving)
drug_z_u = drug_z_u.cpu()
prot_z_u = prot_z_u.cpu()

smiles2emb = {smi: drug_z_u[i] for i, smi in enumerate(unique_smiles)}
protein2emb = {prot: prot_z_u[i] for i, prot in enumerate(unique_proteins)}

In [12]:
torch.save(smiles2emb, '/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_lora_esm2only_smile_embeddings.pt')
torch.save(protein2emb, '/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_lora_esm2only_protein_embeddings.pt')

## LoRA Structure Stage 1 Embeddings (1230_0: ChemBERTa and ESM-2 are trained with LoRA)

In [2]:
def load_dti_model_for_inference(ckpt_path, device="cuda"):

    chem_tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-10M-MLM")
    chem_model = AutoModel.from_pretrained("DeepChem/ChemBERTa-77M-MTR", trust_remote_code=True, use_safetensors=True)
    chem_targets = ["attention.self.query", "attention.self.key", "attention.self.value", "attention.output.dense", "intermediate.dense", "output.dense"]

    chem_model = inject_lora_into_chembert(
        chem_model,
        target_substrings=chem_targets,
        r=4,
        alpha=16,
        dropout=0.05,
        layers_last_k=3,
        freeze_embeddings=True,
        freeze_pooler=True,
    )

    chem_hidden = chem_model.config.hidden_size
    chem_model = make_hf_lora_only(chem_model)
    chem_model.to(device)
    
    esm_model, alphabet = esm.pretrained.esm2_t30_150M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    esm_repr_layer = esm_model.num_layers  # = 30
    esm_hidden = esm_model.embed_dim        # = 640

    esm_model = inject_lora_into_esm(
        esm_model,
        target_substrings=["q_proj", "k_proj", "v_proj", "out_proj"],
        r=4,
        alpha=16,
        dropout=0.05,
        layers_last_k=5,   
    )
    esm_model = make_hf_lora_only(esm_model)
    esm_model.to(device)

    # -------------------------
    # 4) Build DTIContrastiveModel
    # -------------------------
    dti_model = DTIContrastiveModel(
        chem_model=chem_model,
        esm_model=esm_model,
        alphabet=alphabet,
        chem_hidden=chem_hidden,
        esm_hidden=esm_hidden,
        esm_repr_layer=esm_repr_layer,
        proj_hidden=1024,  
        proj_out=512,      
        debug_assert_valid_tokens=False,
        use_cls_pool_for_protein=True, 
    )

    # -------------------------
    # 5) Load checkpoint
    # -------------------------
    state = torch.load(ckpt_path, map_location="cpu")
    dti_model.load_state_dict(state, strict=True)

    # -------------------------
    # 6) Move to device + eval
    # -------------------------
    dti_model = dti_model.to(device)
    dti_model.eval()

    return dti_model, alphabet, batch_converter

In [4]:
def inject_lora_into_chembert(
    chem_model: nn.Module,
    target_substrings: List[str],
    r: int = 4,
    alpha: int = 16,
    dropout: float = 0.05,
    layers_last_k: Optional[int] = None,
    freeze_embeddings: bool = True,
    freeze_pooler: bool = True,
) -> nn.Module:

    # 1) Freeze everything first
    for p in chem_model.parameters():
        p.requires_grad = False

    # Optionally keep embeddings frozen (recommended)
    if not freeze_embeddings and hasattr(chem_model, "embeddings"):
        for p in chem_model.embeddings.parameters():
            p.requires_grad = True  # only if you want non-LoRA embedding tuning (usually no)

    # Optionally keep pooler frozen (recommended)
    if not freeze_pooler and hasattr(chem_model, "pooler") and chem_model.pooler is not None:
        for p in chem_model.pooler.parameters():
            p.requires_grad = True  # only if you want non-LoRA pooler tuning

    # 2) Restrict which encoder layers are eligible (last-k)
    allowed_layer_prefixes = None
    if layers_last_k is not None:
        if hasattr(chem_model, "encoder") and hasattr(chem_model.encoder, "layer") and isinstance(chem_model.encoder.layer, nn.ModuleList):
            L = len(chem_model.encoder.layer)
            start = max(0, L - layers_last_k)
            allowed_layer_prefixes = [f"encoder.layer.{i}." for i in range(start, L)]
        else:
            print("[WARN] layers_last_k provided but chem_model has no encoder.layer ModuleList; ignoring.")

    # 3) Replace targeted Linear layers by LoRALinear
    replaced = 0
    for name, module in list(chem_model.named_modules()):
        if isinstance(module, LoRALinear):
            continue
        if not isinstance(module, nn.Linear):
            continue

        if allowed_layer_prefixes is not None:
            if not any(name.startswith(pref) for pref in allowed_layer_prefixes):
                continue

        if any(s in name for s in target_substrings):
            parent, attr = _get_parent_module(chem_model, name)
            wrapped = LoRALinear(module, r=r, alpha=alpha, dropout=dropout)
            wrapped = wrapped.to(device=module.weight.device, dtype=module.weight.dtype)

            if attr.isdigit():
                parent[int(attr)] = wrapped
            else:
                setattr(parent, attr, wrapped)

            replaced += 1

    if replaced == 0:
        print("[WARN] No Linear layers matched for ChemBERTa LoRA injection.")
        print("[INFO] Available Linear module names:")
        for n, m in chem_model.named_modules():
            if isinstance(m, nn.Linear):
                print(f"  - {n}")
    else:
        print(f"[INFO] Injected LoRA into {replaced} Linear layers of ChemBERTa.")

    return chem_model

In [6]:
class LoRALinear(nn.Module):
    """
    Wrap a frozen nn.Linear with a trainable low-rank update:
      y = x W^T + b + (alpha/r) * x (BA)^T
    """
    def __init__(self, base: nn.Linear, r: int = 4, alpha: int = 16, dropout: float = 0.05):
        super().__init__()
        assert isinstance(base, nn.Linear)

        self.base = base
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze base weights
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # LoRA matrices
        self.A = nn.Linear(self.in_features, r, bias=False)
        self.B = nn.Linear(r, self.out_features, bias=False)

        # Initialization (standard LoRA)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

        self.to(device=base.weight.device, dtype=base.weight.dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = self.B(self.A(self.dropout(x))) * self.scaling
        return base_out + lora_out

In [7]:
def inject_lora_into_esm(
    esm_model: nn.Module,
    target_substrings: List[str],
    r: int = 4,
    alpha: int = 16,
    dropout: float = 0.05,
    layers_last_k: Optional[int] = None,
) -> nn.Module:

    for p in esm_model.parameters():
        p.requires_grad = False

    allowed_layer_prefixes = None
    if layers_last_k is not None:
        if hasattr(esm_model, "layers") and isinstance(esm_model.layers, nn.ModuleList):
            L = len(esm_model.layers)
            start = max(0, L - layers_last_k)
            allowed_layer_prefixes = [f"layers.{i}." for i in range(start, L)]
        else:
            print("[WARN] layers_last_k provided but esm_model has no .layers ModuleList; ignoring.")


    replaced = 0

    for name, module in list(esm_model.named_modules()):

        if isinstance(module, LoRALinear):
            continue

        if not isinstance(module, nn.Linear):
            continue

        if allowed_layer_prefixes is not None:
            if not any(name.startswith(pref) for pref in allowed_layer_prefixes):
                continue

        if any(s in name for s in target_substrings):
            parent, attr = _get_parent_module(esm_model, name)
            wrapped = LoRALinear(module, r=r, alpha=alpha, dropout=dropout)
            wrapped = wrapped.to(device=module.weight.device, dtype=module.weight.dtype)

            if attr.isdigit():
                parent[int(attr)] = wrapped
            else:
                setattr(parent, attr, wrapped)

            replaced += 1


    if replaced == 0:
        print("[WARN] No Linear layers matched for LoRA injection.")
        print("[INFO] Available module names:")
        for n, m in esm_model.named_modules():
            if isinstance(m, nn.Linear):
                print(f"  - {n}")
    else:
        print(f"[INFO] Injected LoRA into {replaced} Linear layers of ESM.")

    return esm_model

In [9]:
def _get_parent_module(root: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = root
    for p in parts[:-1]:
        if p.isdigit():
            parent = parent[int(p)]
        else:
            parent = getattr(parent, p)
    return parent, parts[-1]

In [11]:
def make_hf_lora_only(model: nn.Module) -> nn.Module:
    for p in model.parameters():
        p.requires_grad = False

    for n, p in model.named_parameters():
        if ".A." in n or ".B." in n:
            p.requires_grad = True

    bad = [n for n, p in model.named_parameters()
           if p.requires_grad and not (".A." in n or ".B." in n)]
    if bad:
        print("Still-trainable (should be frozen):", bad[:20])
    else:
        print("✅ Model is LoRA-only now.")
    return model

In [13]:
class ProjectorMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 1024, out_dim: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class DTIContrastiveModel(nn.Module):
    """
    Two-tower encoders (ChemBERTa + ESM2) -> pooled embeddings -> projectors -> normalized z's.

    Key fixes vs your earlier version:
      - alphabet and esm_repr_layer are passed to __init__ (no globals / no NameError risk)
      - forward() does NOT take alphabet/esm_repr_layer/esm_padding_mask (no redundancy / fewer bugs)
      - padding mask is computed inside forward() from esm_tokens using alphabet.padding_idx
      - pooling excludes PAD + CLS + EOS safely (EOS anywhere, even if truncated)
      - optional debug assert to catch empty valid masks
    """
    def __init__(
        self,
        chem_model: nn.Module,
        esm_model: nn.Module,
        alphabet,
        chem_hidden: int,
        esm_hidden: int,
        esm_repr_layer: int = 30,
        proj_hidden: int = 1024,
        proj_out: int = 512,
        debug_assert_valid_tokens: bool = False,
        use_cls_pool_for_protein: bool = True,
    ):
        super().__init__()
        self.chem_model = chem_model
        self.esm_model = esm_model
        self.alphabet = alphabet
        self.esm_repr_layer = esm_repr_layer

        self.debug_assert_valid_tokens = debug_assert_valid_tokens
        self.use_cls_pool_for_protein = use_cls_pool_for_protein

        self.drug_proj = ProjectorMLP(chem_hidden, proj_hidden, proj_out)
        self.prot_proj = ProjectorMLP(esm_hidden, proj_hidden, proj_out)

    def forward(
        self,
        chem_inputs: Dict[str, torch.Tensor],
        esm_tokens: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        
        # -----------------
        # ChemBERTa forward
        # -----------------
        chem_out = self.chem_model(**chem_inputs)
        # RoBERTa-like models expose last_hidden_state: [B, T, H]
        drug_emb = chem_out.last_hidden_state[:, 0, :]  # CLS pooling

        # ------------
        # ESM2 forward
        # ------------
        esm_out = self.esm_model(esm_tokens, repr_layers=[self.esm_repr_layer], return_contacts=False)
        reps = esm_out["representations"][self.esm_repr_layer]  # [B, T, H]

        # Pool protein representation
        if self.use_cls_pool_for_protein:
            # Simple + padding-safe (but sometimes slightly worse than mean pooling)
            prot_emb = reps[:, 0, :]
        else:
            # Mean pool over real residues: exclude PAD + CLS + EOS
            pad_mask = esm_tokens.eq(self.alphabet.padding_idx)  # True for PAD
            mask = ~pad_mask                                      # True for non-PAD
            mask[:, 0] = False                                    # drop CLS token
            mask &= ~esm_tokens.eq(self.alphabet.eos_idx)         # drop EOS anywhere

            if self.debug_assert_valid_tokens and self.training:
                # Ensure each sequence has at least one valid residue token
                if not torch.all(mask.sum(dim=1) > 0):
                    bad = (mask.sum(dim=1) == 0).nonzero(as_tuple=True)[0].tolist()
                    raise ValueError(f"Found {len(bad)} sequences with 0 valid tokens after masking. Indices: {bad[:10]}")

            denom = mask.sum(dim=1).clamp(min=1).unsqueeze(-1)    # [B,1]
            prot_emb = (reps * mask.unsqueeze(-1)).sum(dim=1) / denom  # [B,H]

        # --------------------
        # Project & normalize
        # --------------------
        drug_z = F.normalize(self.drug_proj(drug_emb), dim=1)
        prot_z = F.normalize(self.prot_proj(prot_emb), dim=1)

        return drug_z, prot_z

In [14]:
ckpt_path = "/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/result/1230_0/best_model.pt"
device = "cuda"
dti_model, alphabet, batch_converter = load_dti_model_for_inference(ckpt_path, device=device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] Injected LoRA into 18 Linear layers of ChemBERTa.
✅ Model is LoRA-only now.
[INFO] Injected LoRA into 20 Linear layers of ESM.
✅ Model is LoRA-only now.


/scratch/slurm-1565638/ipykernel_4006711/147527850.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


In [15]:
smile_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/SMILE_embedding.pt", map_location=device)
protein_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrastive_learning/data/embeddings/protein_embedding.pt", map_location=device)

unique_smiles = list(smile_dict.keys())
unique_proteins = list(protein_dict.keys())

/scratch/slurm-1565638/ipykernel_4006711/2802878398.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  smile_dict = torch.load("/ix1/ychiu/yil346/DTI_v2/Structure/Contrasti

In [16]:
@torch.no_grad()
def embed_all_smiles(
    dti_model,
    smiles_list,
    chem_tokenizer,
    device="cuda",
    batch_size=256,
    max_drug_tokens=1024,
    use_amp=True,
):
    dti_model.eval()
    use_cuda = device.startswith("cuda")
    out = []

    for i in tqdm(range(0, len(smiles_list), batch_size), desc="Embed SMILES"):
        batch = smiles_list[i:i+batch_size]

        chem_inputs = chem_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_drug_tokens,
        )
        chem_inputs = {k: v.to(device) for k, v in chem_inputs.items()}

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(use_amp and use_cuda)):
            chem_out = dti_model.chem_model(**chem_inputs)
            drug_emb = chem_out.last_hidden_state[:, 0, :]  # CLS
            drug_z = F.normalize(dti_model.drug_proj(drug_emb), dim=1)

        out.append(drug_z.cpu())

    return torch.cat(out, dim=0)

In [18]:
chem_tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-10M-MLM")

drug_z_u = embed_all_smiles(
    dti_model, unique_smiles, chem_tokenizer,
    device=device,
    batch_size=128,
    max_drug_tokens=1024,
)
print(drug_z_u.shape)

Embed SMILES: 100%|██████████| 565/565 [00:11<00:00, 49.49it/s]

torch.Size([72227, 512])


In [19]:
@torch.no_grad()
def embed_all_proteins(
    dti_model,
    protein_list,
    batch_converter,
    device="cuda",
    batch_size=32,
    max_protein_tokens=1024,
    use_amp=True,
):
    dti_model.eval()
    use_cuda = device.startswith("cuda")
    out = []

    for i in tqdm(range(0, len(protein_list), batch_size), desc="Embed proteins"):
        batch = protein_list[i:i+batch_size]

        esm_data = [(str(j), seq) for j, seq in enumerate(batch)]
        _, _, tokens = batch_converter(esm_data)

        # truncate defensively
        if tokens.size(1) > max_protein_tokens:
            tokens = tokens[:, :max_protein_tokens]

        tokens = tokens.to(device)

        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(use_amp and use_cuda)):
            esm_out = dti_model.esm_model(tokens, repr_layers=[dti_model.esm_repr_layer], return_contacts=False)
            reps = esm_out["representations"][dti_model.esm_repr_layer]

            # CLS pooling (matches your training)
            prot_emb = reps[:, 0, :]
            prot_z = F.normalize(dti_model.prot_proj(prot_emb), dim=1)

        out.append(prot_z.cpu())

    return torch.cat(out, dim=0)

In [20]:
prot_z_u = embed_all_proteins(
    dti_model, unique_proteins, batch_converter,
    device=device,
    batch_size=8,  # keep small for ESM
    max_protein_tokens=1024,
)
print(prot_z_u.shape)

Embed proteins: 100%|██████████| 1007/1007 [03:40<00:00,  4.57it/s]

torch.Size([8054, 512])


In [21]:
# Ensure CPU tensors (recommended for saving)
drug_z_u = drug_z_u.cpu()
prot_z_u = prot_z_u.cpu()

smiles2emb = {smi: drug_z_u[i] for i, smi in enumerate(unique_smiles)}
protein2emb = {prot: prot_z_u[i] for i, prot in enumerate(unique_proteins)}

In [22]:
torch.save(smiles2emb, '/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_lora_both_smile_embeddings.pt')
torch.save(protein2emb, '/ix1/ychiu/yil346/DTI_v2/Unified/contrastive_learning/data/embeddings/contrastive_new_lora_both_protein_embeddings.pt')